<center><strong>Deep Learning : Computer Vision</strong><br />
<strong><font color="blue">Departemen Informatika - Prodi RKA - Institut Teknologi Sepuluh Nopember</font></strong><br />
</center>

<strong>Outline Hands-On: Image Segmentation with U-Net</strong><br />
<li> Konsep Image Segmentation & Perbedaan dengan Task CV Lainnya </li>
<li> Arsitektur U-Net (Encoder-Decoder + Skip Connections) </li>
<li> Implementasi U-Net dari Scratch dengan Dataset Oxford-IIIT Pet </li>
<li> Transfer Learning untuk Segmentation: U-Net + MobileNetV2 Backbone </li>
<li> Data Augmentation untuk Segmentation </li>
<li> Evaluasi Mendalam: IoU, Dice Coefficient & Analisis Per-Kelas </li>

---

## Setup & Persiapan Environment

<p><strong><font color="red">PENTING:</font></strong> Pastikan runtime Google Colab menggunakan <strong>GPU</strong> untuk mempercepat proses training.</p>

<p>Cara mengaktifkan GPU di Google Colab:</p>
<ol>
    <li>Klik menu <strong>Runtime</strong> → <strong>Change runtime type</strong></li>
    <li>Pada <strong>Hardware accelerator</strong>, pilih <strong>GPU (T4)</strong></li>
    <li>Klik <strong>Save</strong></li>
</ol>

In [ ]:
# Import library yang dibutuhkan
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2

import tensorflow_datasets as tfds

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Fungsi helper untuk memplot training history
def plot_history(history, title=''):
    """Plot accuracy & loss dari training history."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history.history['accuracy'], 'b-', label='Training', linewidth=2)
    ax1.plot(history.history['val_accuracy'], 'r-', label='Validation', linewidth=2)
    ax1.set_title(f'Accuracy History {title}', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history.history['loss'], 'b-', label='Training', linewidth=2)
    ax2.plot(history.history['val_loss'], 'r-', label='Validation', linewidth=2)
    ax2.set_title(f'Loss History {title}', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

---
## 1. Image Segmentation: Konsep & Semantic Segmentation

<p><strong>Image Segmentation</strong> adalah task Computer Vision yang bertujuan untuk <strong>mengklasifikasikan setiap piksel</strong> dalam sebuah gambar ke dalam kategori tertentu — bukan hanya seluruh gambar (Classification) atau kotak bounding box (Detection).</p>

### Perbedaan Task Computer Vision:

<table border="1" cellpadding="8" cellspacing="0">
<tr><th>Task</th><th>Output</th><th>Contoh</th></tr>
<tr><td><strong>Classification</strong></td><td>1 label untuk seluruh gambar</td><td>"Kucing"</td></tr>
<tr><td><strong>Object Detection</strong></td><td>Label + koordinat bounding box</td><td>"Kucing" di [x1,y1,x2,y2]</td></tr>
<tr><td><strong>Semantic Segmentation</strong></td><td>Label per-piksel (semua objek sama kelas = warna sama)</td><td>Semua piksel kucing = warna kuning</td></tr>
<tr><td><strong>Instance Segmentation</strong></td><td>Label + mask per-objek individual</td><td>Kucing-1 = merah, Kucing-2 = biru</td></tr>
</table>

### Arsitektur Utama untuk Segmentation:

<ol>
    <li><strong>FCN (Fully Convolutional Network)</strong> — Pertama mengganti Dense layer dengan Conv layer, sehingga bisa menghasilkan output berupa spatial map</li>
    <li><strong>U-Net</strong> — Arsitektur <em>encoder-decoder</em> dengan <em>skip connections</em>; sangat populer untuk medical imaging</li>
    <li><strong>DeepLab</strong> — Menggunakan <em>dilated/atrous convolution</em> untuk memperluas receptive field tanpa mengurangi resolusi</li>
    <li><strong>Mask R-CNN</strong> — Ekstensi dari Faster R-CNN untuk instance segmentation</li>
</ol>

### Arsitektur U-Net:

<p>U-Net terdiri dari dua bagian utama:</p>
<ul>
    <li><strong>Encoder (Contracting Path)</strong> — Seperti CNN biasa: Conv + MaxPool secara berulang → gambar mengecil, fitur semakin abstrak</li>
    <li><strong>Decoder (Expanding Path)</strong> — Kebalikannya: UpSampling/Transpose Conv untuk membesarkan kembali ke ukuran asli</li>
    <li><strong>Skip Connections</strong> — Menghubungkan langsung layer encoder ke decoder pada resolusi yang sama → mempertahankan detail spasial</li>
</ul>

```
Input → [Encoder] → Bottleneck → [Decoder] → Output Mask
            ↑___________skip connections__________↑
```


In [ ]:
# Visualisasi konsep perbedaan task Computer Vision
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Perbandingan Task Computer Vision', fontsize=15, fontweight='bold', y=1.02)

# Buat gambar sederhana sebagai ilustrasi (2 objek: persegi panjang = kucing & anjing)
base_img = np.ones((100, 150, 3)) * 0.85  # background abu-abu
base_img[20:70, 15:65] = [0.9, 0.75, 0.5]   # objek 1 (kucing) - oranye
base_img[20:70, 85:135] = [0.5, 0.7, 0.9]   # objek 2 (anjing) - biru

# --- 1. Classification ---
axes[0].imshow(base_img)
axes[0].set_title('1. Classification\n→ "Kucing"', fontsize=11, fontweight='bold')
axes[0].text(75, 110, 'Label: Kucing', ha='center', fontsize=10,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
axes[0].axis('off')

# --- 2. Object Detection ---
axes[1].imshow(base_img)
axes[1].set_title('2. Object Detection\n→ Label + Bounding Box', fontsize=11, fontweight='bold')
rect1 = mpatches.Rectangle((15, 20), 50, 50, linewidth=2.5, edgecolor='red', facecolor='none')
rect2 = mpatches.Rectangle((85, 20), 50, 50, linewidth=2.5, edgecolor='blue', facecolor='none')
axes[1].add_patch(rect1)
axes[1].add_patch(rect2)
axes[1].text(40, 17, 'Kucing', color='red', fontsize=9, fontweight='bold', ha='center')
axes[1].text(110, 17, 'Anjing', color='blue', fontsize=9, fontweight='bold', ha='center')
axes[1].axis('off')

# --- 3. Semantic Segmentation ---
sem_img = np.ones((100, 150, 3)) * 0.3  # background gelap
sem_img[20:70, 15:65] = [0.9, 0.75, 0.5]   # kedua objek warna SAMA (sama-sama hewan)
sem_img[20:70, 85:135] = [0.9, 0.75, 0.5]  # warna sama karena kelas sama
axes[2].imshow(sem_img)
axes[2].set_title('3. Semantic Segmentation\n→ Label per-piksel (kelas sama = warna sama)', fontsize=11, fontweight='bold')
axes[2].text(75, 110, 'Semua "hewan" = warna sama', ha='center', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
axes[2].axis('off')

# --- 4. Instance Segmentation ---
inst_img = np.ones((100, 150, 3)) * 0.3
inst_img[20:70, 15:65] = [0.9, 0.4, 0.4]   # objek 1 = merah
inst_img[20:70, 85:135] = [0.4, 0.6, 0.9]  # objek 2 = biru (berbeda walau kelas sama)
axes[3].imshow(inst_img)
axes[3].set_title('4. Instance Segmentation\n→ Mask per-objek individual', fontsize=11, fontweight='bold')
axes[3].text(40, 80, 'Kucing-1', color='white', fontsize=9, ha='center')
axes[3].text(110, 80, 'Anjing-1', color='white', fontsize=9, ha='center')
axes[3].axis('off')

plt.tight_layout()
plt.show()
print("Catatan: Semantic Segmentation = semua objek kelas yang sama → warna sama")
print("         Instance Segmentation = setiap objek individual → warna berbeda")


### Implementasi U-Net Sederhana

<p>Kita akan mengimplementasikan U-Net sederhana menggunakan dataset <strong>Oxford-IIIT Pet</strong> yang sudah tersedia di TensorFlow Datasets. Dataset ini berisi:</p>
<ul>
    <li>~7.400 gambar hewan peliharaan (37 jenis kucing & anjing)</li>
    <li>Setiap gambar sudah memiliki <strong>segmentation mask</strong> (3 kelas: objek, background, border)</li>
</ul>

<p><strong>Arsitektur U-Net yang akan kita bangun:</strong></p>
<pre>
Input (128×128×3)
  ↓
[Encoder]
  Conv(64) → Conv(64) → MaxPool  → skip1 (128×128×64)
  Conv(128) → Conv(128) → MaxPool → skip2 (64×64×128)
  Conv(256) → Conv(256) → MaxPool → skip3 (32×32×256)
  ↓
[Bottleneck]
  Conv(512) → Conv(512)           (16×16×512)
  ↓
[Decoder]
  UpSample + skip3 → Conv(256) → Conv(256)  (32×32×256)
  UpSample + skip2 → Conv(128) → Conv(128)  (64×64×128)
  UpSample + skip1 → Conv(64)  → Conv(64)   (128×128×64)
  ↓
Output Conv(3, softmax)          (128×128×3 → 3 kelas)
</pre>


In [ ]:
# Install tensorflow-datasets jika belum tersedia
# !pip install tensorflow-datasets -q

import tensorflow_datasets as tfds
import tensorflow as tf

# Load Oxford-IIIT Pet dataset
print("Mengunduh dataset Oxford-IIIT Pet...")
dataset, info = tfds.load('oxford_iiit_pet:4.0.0', with_info=True)  # ← ganti 3.*.* jadi 4.0.0

print(f"\nInfo dataset:")
print(f"  Splits tersedia : {list(dataset.keys())}")
print(f"  Jumlah train    : {info.splits['train'].num_examples}")
print(f"  Jumlah test     : {info.splits['test'].num_examples}")
print(f"  Fitur           : {info.features}")

In [ ]:
# Preprocessing: resize gambar & mask ke 128x128
IMG_SIZE = 128
BATCH_SIZE = 32

def preprocess(datapoint):
    """Resize gambar dan mask, normalisasi gambar."""
    # Gambar input: resize ke 128x128, normalisasi ke [0,1]
    image = tf.image.resize(datapoint['image'], (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0

    # Mask: resize ke 128x128
    # Label Oxford Pet: 1=foreground(hewan), 2=background, 3=border
    # Kita shift ke 0-based: 0=hewan, 1=background, 2=border
    mask = tf.image.resize(datapoint['segmentation_mask'], (IMG_SIZE, IMG_SIZE),
                           method='nearest')  # nearest agar label tidak interpolasi
    mask = tf.cast(mask, tf.int32) - 1  # shift: [1,2,3] → [0,1,2]

    return image, mask

# Build dataset pipeline
train_dataset = (dataset['train']
                 .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
                 .shuffle(1000)
                 .batch(BATCH_SIZE)
                 .prefetch(tf.data.AUTOTUNE))

test_dataset = (dataset['test']
                .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
                .batch(BATCH_SIZE)
                .prefetch(tf.data.AUTOTUNE))

print("Dataset siap!")
print(f"Contoh shape: image={list(train_dataset)[0][0].shape}, mask={list(train_dataset)[0][1].shape}")


In [ ]:
# Visualisasi contoh gambar dan mask-nya
class_labels = ['Hewan (foreground)', 'Background', 'Border']
class_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']  # merah, hijau, biru

sample_images, sample_masks = next(iter(train_dataset))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i in range(3):
    # Gambar asli
    axes[i, 0].imshow(sample_images[i])
    axes[i, 0].set_title('Gambar Asli', fontsize=11)
    axes[i, 0].axis('off')

    # Mask ground truth
    axes[i, 1].imshow(sample_masks[i, :, :, 0], cmap='tab10', vmin=0, vmax=2)
    axes[i, 1].set_title('Mask (Ground Truth)\n0=Hewan, 1=BG, 2=Border', fontsize=11)
    axes[i, 1].axis('off')

    # Overlay: gambar + mask
    mask_rgb = np.zeros((*sample_masks[i, :, :, 0].shape, 3))
    mask_rgb[sample_masks[i, :, :, 0] == 0] = [1, 0.4, 0.4]    # merah = hewan
    mask_rgb[sample_masks[i, :, :, 0] == 1] = [0.3, 0.8, 0.75] # hijau = background
    mask_rgb[sample_masks[i, :, :, 0] == 2] = [0.3, 0.7, 1.0]  # biru = border

    overlay = sample_images[i].numpy() * 0.6 + mask_rgb * 0.4
    axes[i, 2].imshow(np.clip(overlay, 0, 1))
    axes[i, 2].set_title('Overlay\n(Gambar + Mask)', fontsize=11)
    axes[i, 2].axis('off')

    # Distribusi piksel per kelas
    mask_flat = sample_masks[i, :, :, 0].numpy().flatten()
    counts = [np.sum(mask_flat == c) for c in range(3)]
    bars = axes[i, 3].bar(class_labels, counts,
                          color=['#FF6B6B', '#4ECDC4', '#45B7D1'], edgecolor='black')
    axes[i, 3].set_title('Distribusi Piksel', fontsize=11)
    axes[i, 3].set_ylabel('Jumlah piksel')
    axes[i, 3].tick_params(axis='x', rotation=15)
    for bar, count in zip(bars, counts):
        axes[i, 3].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                        f'{count:,}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Contoh Data Oxford-IIIT Pet: Gambar, Mask, Overlay, Distribusi',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### Membangun Arsitektur U-Net

<p>Kita menggunakan <strong>Functional API</strong> Keras (bukan Sequential) karena U-Net memiliki <em>skip connections</em> — koneksi langsung dari encoder ke decoder yang tidak bisa direpresentasikan secara sequential.</p>

In [ ]:
from tensorflow.keras import layers, Model

def conv_block(x, filters, kernel_size=3):
    """Blok 2x Conv2D + ReLU — unit dasar encoder & decoder U-Net."""
    x = layers.Conv2D(filters, kernel_size, padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, kernel_size, padding='same', activation='relu')(x)
    return x

def build_unet(input_shape=(128, 128, 3), num_classes=3):
    """Membangun arsitektur U-Net dengan skip connections."""
    inputs = layers.Input(shape=input_shape)

    # ── ENCODER (Contracting Path) ───────────────────────────────────────
    # Setiap level: 2x Conv → simpan skip → MaxPool (ukuran mengecil ½)
    e1 = conv_block(inputs, 64)          # output: 128×128×64
    p1 = layers.MaxPooling2D(2)(e1)      # output: 64×64×64

    e2 = conv_block(p1, 128)             # output: 64×64×128
    p2 = layers.MaxPooling2D(2)(e2)      # output: 32×32×128

    e3 = conv_block(p2, 256)             # output: 32×32×256
    p3 = layers.MaxPooling2D(2)(e3)      # output: 16×16×256

    # ── BOTTLENECK ────────────────────────────────────────────────────────
    b = conv_block(p3, 512)              # output: 16×16×512

    # ── DECODER (Expanding Path) ──────────────────────────────────────────
    # Setiap level: UpSampling2D (ukuran membesar 2x) → gabung skip → 2x Conv
    d1 = layers.UpSampling2D(2)(b)              # 16×16 → 32×32
    d1 = layers.Concatenate()([d1, e3])         # gabung dengan skip e3
    d1 = conv_block(d1, 256)                    # output: 32×32×256

    d2 = layers.UpSampling2D(2)(d1)             # 32×32 → 64×64
    d2 = layers.Concatenate()([d2, e2])         # gabung dengan skip e2
    d2 = conv_block(d2, 128)                    # output: 64×64×128

    d3 = layers.UpSampling2D(2)(d2)             # 64×64 → 128×128
    d3 = layers.Concatenate()([d3, e1])         # gabung dengan skip e1
    d3 = conv_block(d3, 64)                     # output: 128×128×64

    # ── OUTPUT LAYER ──────────────────────────────────────────────────────
    # 1×1 Conv untuk mapping ke jumlah kelas (per-piksel classification)
    outputs = layers.Conv2D(num_classes, 1, activation='softmax')(d3)  # 128×128×3

    return Model(inputs, outputs, name='U-Net')

model_unet = build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=3)
model_unet.summary()


In [ ]:
# Compile U-Net
# sparse_categorical_crossentropy → label berupa integer (bukan one-hot)
model_unet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Early stopping agar tidak overfitting
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

print("Memulai training U-Net...")
history_unet = model_unet.fit(
    train_dataset,
    epochs=10,
    validation_data=test_dataset,
    callbacks=[early_stop],
    verbose=1
)

# Evaluasi
test_loss_unet, test_acc_unet = model_unet.evaluate(test_dataset, verbose=0)
print(f"\nTest Accuracy U-Net: {test_acc_unet:.4f}")


In [ ]:
# Plot training history
plot_history(history_unet, '- U-Net Semantic Segmentation')


In [ ]:
# Visualisasi hasil prediksi U-Net
def visualize_segmentation(model, dataset, num_samples=4):
    """Tampilkan perbandingan: Gambar Asli | Ground Truth | Prediksi | Overlay."""
    sample_images, sample_masks = next(iter(dataset))
    predictions = model.predict(sample_images[:num_samples], verbose=0)
    pred_masks = np.argmax(predictions, axis=-1)  # ambil kelas dengan prob tertinggi

    fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples * 4))
    col_titles = ['Gambar Asli', 'Ground Truth', 'Prediksi U-Net', 'Overlay Prediksi']

    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=12, fontweight='bold')

    def mask_to_rgb(mask_2d):
        """Konversi mask integer ke RGB untuk visualisasi."""
        rgb = np.zeros((*mask_2d.shape, 3))
        rgb[mask_2d == 0] = [1.0, 0.4, 0.4]    # merah  = hewan
        rgb[mask_2d == 1] = [0.3, 0.8, 0.75]   # hijau  = background
        rgb[mask_2d == 2] = [0.3, 0.7, 1.0]    # biru   = border
        return rgb

    for i in range(num_samples):
        img = sample_images[i].numpy()
        gt_mask = sample_masks[i, :, :, 0].numpy()
        pred_mask = pred_masks[i]

        # Gambar asli
        axes[i, 0].imshow(img)
        axes[i, 0].axis('off')

        # Ground truth
        axes[i, 1].imshow(mask_to_rgb(gt_mask))
        axes[i, 1].axis('off')

        # Prediksi
        axes[i, 2].imshow(mask_to_rgb(pred_mask))
        axes[i, 2].axis('off')

        # Overlay prediksi di atas gambar asli
        overlay = img * 0.55 + mask_to_rgb(pred_mask) * 0.45
        axes[i, 3].imshow(np.clip(overlay, 0, 1))
        axes[i, 3].axis('off')

    # Legend
    legend_patches = [
        mpatches.Patch(color='#FF6B6B', label='Hewan (foreground)'),
        mpatches.Patch(color='#4ECDC4', label='Background'),
        mpatches.Patch(color='#45B7D1', label='Border'),
    ]
    fig.legend(handles=legend_patches, loc='lower center', ncol=3,
               fontsize=11, bbox_to_anchor=(0.5, -0.02))

    plt.suptitle('Hasil Segmentasi U-Net pada Oxford-IIIT Pet Dataset',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_segmentation(model_unet, test_dataset, num_samples=4)


<h3><font color="blue">Latihan 1:</font></h3>

<ul>
    <li>Perhatikan arsitektur U-Net yang sudah dibangun. <strong>Apa fungsi dari <code>layers.Concatenate()</code></strong> pada bagian decoder? Apa yang terjadi jika skip connections dihapus?</li>
    <li>Coba tambahkan satu level encoder-decoder lagi (Conv(512) → MaxPool, lalu UpSample + skip → Conv(512)). Hitung berapa jumlah parameter yang bertambah.</li>
    <li>Mengapa pada segmentation kita menggunakan <code>sparse_categorical_crossentropy</code> bukan <code>categorical_crossentropy</code> seperti di klasifikasi sebelumnya?</li>
</ul>

<p><strong>Hint:</strong> <code>categorical_crossentropy</code> membutuhkan label dalam format one-hot encoding, sedangkan <code>sparse_categorical_crossentropy</code> menerima label berupa integer langsung.</p>


In [ ]:
# Kerjakan Latihan 1 pada cell berikut ini



---
## 2. Image Segmentation: Demo Pre-trained Model (MobileNetV2 + U-Net)

<p>Melatih U-Net dari scratch membutuhkan waktu dan data yang banyak. Dalam praktiknya, kita bisa menggunakan <strong>Transfer Learning</strong> juga untuk segmentation — dengan memakai pre-trained model sebagai <strong>encoder</strong>, lalu hanya melatih bagian <strong>decoder</strong>-nya.</p>

<p>Teknik ini disebut <strong>Encoder-Decoder with Pre-trained Backbone</strong>:</p>

<ul>
    <li><strong>Encoder (Backbone)</strong> — MobileNetV2 pre-trained ImageNet → sudah bisa mengekstrak fitur</li>
    <li><strong>Decoder</strong> — Dilatih dari awal untuk memprediksi mask segmentasi</li>
    <li>Skip connections diambil dari layer-layer tengah MobileNetV2</li>
</ul>

<p>Arsitektur ini merupakan basis dari model populer seperti <strong>DeepLabV3+</strong> dan <strong>Segmentation Models library</strong>.</p>


In [ ]:
from tensorflow.keras.applications import MobileNetV2

def build_unet_mobilenet(input_shape=(128, 128, 3), num_classes=3):
    """
    U-Net dengan MobileNetV2 sebagai encoder (pre-trained ImageNet).
    Skip connections diambil dari layer-layer spesifik MobileNetV2.
    """
    # ── ENCODER: MobileNetV2 pre-trained ──────────────────────────────────
    backbone = MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    backbone.trainable = False  # Freeze encoder

    # Ambil output dari layer-layer spesifik untuk skip connections
    # Layer dipilih berdasarkan resolusi output-nya
    skip_layer_names = [
        'block_1_expand_relu',   # 64×64
        'block_3_expand_relu',   # 32×32
        'block_6_expand_relu',   # 16×16
        'block_13_expand_relu',  # 8×8
    ]

    # Buat model yang output-nya adalah fitur dari layer-layer tersebut
    skip_outputs = [backbone.get_layer(name).output for name in skip_layer_names]
    encoder_model = Model(inputs=backbone.input,
                          outputs=[backbone.output] + skip_outputs,
                          name='MobileNetV2_Encoder')

    # ── INPUT & FORWARD PASS ENCODER ──────────────────────────────────────
    inputs = layers.Input(shape=input_shape)
    encoder_outputs = encoder_model(inputs)
    x = encoder_outputs[0]                  # bottleneck: 4×4 (atau 8×8)
    skips = encoder_outputs[1:][::-1]       # skip connections (dibalik: dari dalam ke luar)

    # ── DECODER ───────────────────────────────────────────────────────────
    decoder_filters = [512, 256, 128, 64]
    for i, (skip, n_filters) in enumerate(zip(skips, decoder_filters)):
        x = layers.UpSampling2D(2)(x)
        x = layers.Concatenate()([x, skip])
        x = conv_block(x, n_filters)

    # Satu UpSampling lagi agar output = input size
    x = layers.UpSampling2D(2)(x)
    x = conv_block(x, 32)

    # ── OUTPUT ────────────────────────────────────────────────────────────
    outputs = layers.Conv2D(num_classes, 1, activation='softmax')(x)

    return Model(inputs, outputs, name='UNet_MobileNetV2')


model_unet_mob = build_unet_mobilenet(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=3)

# Ringkasan parameter
total    = model_unet_mob.count_params()
trainable = sum([tf.keras.backend.count_params(w) for w in model_unet_mob.trainable_weights])
frozen   = total - trainable
print(f"Total parameter     : {total:,}")
print(f"Trainable (decoder) : {trainable:,}")
print(f"Frozen (encoder)    : {frozen:,}")


In [ ]:
# Compile dan training
model_unet_mob.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop2 = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

print("Training U-Net + MobileNetV2 backbone...")
history_unet_mob = model_unet_mob.fit(
    train_dataset,
    epochs=10,
    validation_data=test_dataset,
    callbacks=[early_stop2],
    verbose=1
)

_, test_acc_mob = model_unet_mob.evaluate(test_dataset, verbose=0)
print(f"\nTest Accuracy (UNet+MobileNetV2): {test_acc_mob:.4f}")


In [ ]:
# Perbandingan: U-Net scratch vs U-Net + MobileNetV2
plot_history(history_unet_mob, '- U-Net + MobileNetV2 Backbone')

print("\n" + "=" * 55)
print("PERBANDINGAN MODEL SEGMENTATION")
print("=" * 55)
print(f"{'Model':<35} {'Test Acc':>10}")
print("-" * 55)
print(f"{'U-Net (dari scratch)':<35} {test_acc_unet:>10.4f}")
print(f"{'U-Net + MobileNetV2 (backbone)':<35} {test_acc_mob:>10.4f}")
print("=" * 55)


In [ ]:
# Visualisasi perbandingan prediksi kedua model
sample_images, sample_masks = next(iter(test_dataset))
n = 3

preds_scratch = np.argmax(model_unet.predict(sample_images[:n], verbose=0), axis=-1)
preds_mob     = np.argmax(model_unet_mob.predict(sample_images[:n], verbose=0), axis=-1)

def mask_to_rgb(mask_2d):
    rgb = np.zeros((*mask_2d.shape, 3))
    rgb[mask_2d == 0] = [1.0, 0.4, 0.4]
    rgb[mask_2d == 1] = [0.3, 0.8, 0.75]
    rgb[mask_2d == 2] = [0.3, 0.7, 1.0]
    return rgb

fig, axes = plt.subplots(n, 4, figsize=(16, n * 4))
col_titles = ['Gambar Asli', 'Ground Truth', 'U-Net (scratch)', 'U-Net + MobileNetV2']

for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=12, fontweight='bold')

for i in range(n):
    axes[i, 0].imshow(sample_images[i].numpy())
    axes[i, 0].axis('off')
    axes[i, 1].imshow(mask_to_rgb(sample_masks[i, :, :, 0].numpy()))
    axes[i, 1].axis('off')
    axes[i, 2].imshow(mask_to_rgb(preds_scratch[i]))
    axes[i, 2].axis('off')
    axes[i, 3].imshow(mask_to_rgb(preds_mob[i]))
    axes[i, 3].axis('off')

legend_patches = [
    mpatches.Patch(color='#FF6B6B', label='Hewan'),
    mpatches.Patch(color='#4ECDC4', label='Background'),
    mpatches.Patch(color='#45B7D1', label='Border'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3,
           fontsize=11, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Perbandingan: U-Net Scratch vs U-Net + MobileNetV2 Backbone',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


<h3><font color="blue">Latihan 2:</font></h3>

<ul>
    <li>Bandingkan <strong>jumlah parameter trainable</strong> antara U-Net scratch dan U-Net + MobileNetV2. Mengapa dengan backbone pre-trained kita perlu melatih lebih sedikit parameter?</li>
    <li>Pada model U-Net + MobileNetV2, coba lakukan <strong>fine-tuning</strong>: unfreeze 20 layer terakhir backbone dengan learning rate yang sangat kecil (<code>1e-5</code>). Apakah akurasi meningkat?</li>
    <li><strong>(Tantangan)</strong> Implementasikan metrik evaluasi <strong>IoU (Intersection over Union)</strong> secara manual untuk membandingkan prediksi dengan ground truth. IoU lebih umum digunakan daripada accuracy dalam segmentation.</li>
</ul>

<p><strong>Hint IoU:</strong></p>

```python
def compute_iou(y_true, y_pred, num_classes=3):
    iou_per_class = []
    for cls in range(num_classes):
        true_cls = (y_true == cls)
        pred_cls = (y_pred == cls)
        intersection = np.logical_and(true_cls, pred_cls).sum()
        union = np.logical_or(true_cls, pred_cls).sum()
        iou = intersection / (union + 1e-7)  # +1e-7 untuk hindari pembagian 0
        iou_per_class.append(iou)
    return np.mean(iou_per_class)  # mean IoU (mIoU)
```

<p><strong>Hint Fine-Tuning MobileNetV2:</strong></p>

```python
# Unfreeze 20 layer terakhir backbone
backbone = model_unet_mob.get_layer('MobileNetV2_Encoder').get_layer('mobilenetv2_1.00_128')
for layer in backbone.layers[-20:]:
    layer.trainable = True
model_unet_mob.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                       loss='sparse_categorical_crossentropy', metrics=['accuracy'])
```


In [ ]:
# Kerjakan Latihan 2 pada cell berikut ini



---
## 3. Data Augmentation untuk Segmentation

<p>Data Augmentation pada segmentation <strong>berbeda</strong> dari klasifikasi. Pada klasifikasi, kita hanya mentransformasi gambar. Pada segmentation, <strong>gambar DAN mask harus ditransformasi secara identik</strong> — jika gambar di-flip horizontal, mask juga harus di-flip horizontal.</p>

<p><strong>Mengapa ini penting?</strong></p>
<ul>
    <li>Jika hanya gambar yang di-augmentasi tanpa mask, maka piksel label tidak lagi sesuai dengan objek di gambar</li>
    <li>Augmentasi geometrik (flip, rotasi, crop) <strong>harus</strong> diterapkan ke keduanya</li>
    <li>Augmentasi warna/brightness <strong>hanya</strong> diterapkan ke gambar (mask tidak terpengaruh warna)</li>
</ul>

<table border="1" cellpadding="8" cellspacing="0">
<tr><th>Jenis Augmentasi</th><th>Diterapkan ke Gambar?</th><th>Diterapkan ke Mask?</th></tr>
<tr><td><strong>Horizontal Flip</strong></td><td>✅ Ya</td><td>✅ Ya</td></tr>
<tr><td><strong>Vertical Flip</strong></td><td>✅ Ya</td><td>✅ Ya</td></tr>
<tr><td><strong>Rotasi</strong></td><td>✅ Ya</td><td>✅ Ya</td></tr>
<tr><td><strong>Random Crop</strong></td><td>✅ Ya</td><td>✅ Ya</td></tr>
<tr><td><strong>Brightness / Contrast</strong></td><td>✅ Ya</td><td>❌ Tidak</td></tr>
<tr><td><strong>Color Jitter / Hue</strong></td><td>✅ Ya</td><td>❌ Tidak</td></tr>
</table>


In [ ]:
# Implementasi Data Augmentation untuk Segmentation
# PENTING: transformasi geometrik harus konsisten antara gambar dan mask!

def augment_segmentation(image, mask):
    """Augmentasi gambar dan mask secara konsisten."""

    # --- Augmentasi GEOMETRIK (diterapkan ke gambar DAN mask) ---

    # 1. Random horizontal flip (50% probabilitas)
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask  = tf.image.flip_left_right(mask)

    # 2. Random vertical flip (30% probabilitas)
    if tf.random.uniform(()) > 0.7:
        image = tf.image.flip_up_down(image)
        mask  = tf.image.flip_up_down(mask)

    # --- Augmentasi WARNA (HANYA diterapkan ke gambar) ---

    # 3. Random brightness
    image = tf.image.random_brightness(image, max_delta=0.2)

    # 4. Random contrast
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)

    # 5. Random saturation
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)

    # Pastikan piksel gambar tetap dalam range [0, 1]
    image = tf.clip_by_value(image, 0.0, 1.0)

    return image, mask

print("Fungsi augment_segmentation siap!")

In [ ]:
# Visualisasi: gambar asli vs augmented (gambar + mask konsisten)
sample_img, sample_msk = next(iter(train_dataset))
orig_img  = sample_img[0]
orig_mask = sample_msk[0]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Data Augmentation untuk Segmentation\n(Baris atas: Gambar, Baris bawah: Mask)',
             fontsize=14, fontweight='bold')

# Kolom 0: Original
axes[0, 0].imshow(orig_img.numpy())
axes[0, 0].set_title('Original', fontsize=11, fontweight='bold')
axes[0, 0].axis('off')
axes[1, 0].imshow(orig_mask[:, :, 0].numpy(), cmap='tab10', vmin=0, vmax=2)
axes[1, 0].set_title('Mask Original', fontsize=11, fontweight='bold')
axes[1, 0].axis('off')

# Kolom 1-4: Augmented versions
for col in range(1, 5):
    aug_img, aug_mask = augment_segmentation(orig_img, orig_mask)
    axes[0, col].imshow(aug_img.numpy())
    axes[0, col].set_title(f'Augmented {col}', fontsize=11)
    axes[0, col].axis('off')
    axes[1, col].imshow(aug_mask[:, :, 0].numpy(), cmap='tab10', vmin=0, vmax=2)
    axes[1, col].set_title(f'Mask Aug {col}', fontsize=11)
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()
print("Perhatikan: saat gambar di-flip, mask juga ikut ter-flip secara konsisten!")

In [ ]:
# Rebuild dataset pipeline DENGAN augmentasi
def preprocess_aug(datapoint):
    """Preprocessing + augmentasi untuk training."""
    image = tf.image.resize(datapoint['image'], (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    mask  = tf.image.resize(datapoint['segmentation_mask'], (IMG_SIZE, IMG_SIZE),
                            method='nearest')
    mask  = tf.cast(mask, tf.int32) - 1
    # Terapkan augmentasi
    image, mask = augment_segmentation(image, mask)
    return image, mask

train_dataset_aug = (dataset['train']
                     .map(preprocess_aug, num_parallel_calls=tf.data.AUTOTUNE)
                     .shuffle(1000)
                     .batch(BATCH_SIZE)
                     .prefetch(tf.data.AUTOTUNE))

print("Dataset dengan augmentasi siap!")

In [ ]:
# Training U-Net (scratch) DENGAN augmentasi
# Rebuild model baru agar perbandingan adil
model_unet_aug = build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=3)

model_unet_aug.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop_aug = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

print("Training U-Net DENGAN augmentasi...")
history_unet_aug = model_unet_aug.fit(
    train_dataset_aug,
    epochs=15,
    validation_data=test_dataset,
    callbacks=[early_stop_aug],
    verbose=1
)

_, test_acc_unet_aug = model_unet_aug.evaluate(test_dataset, verbose=0)
print(f"\nTest Accuracy (U-Net + Augmentasi): {test_acc_unet_aug:.4f}")

In [ ]:
# Perbandingan: U-Net tanpa vs dengan augmentasi
plot_history(history_unet_aug, '- U-Net + Data Augmentation')

print("\n" + "=" * 55)
print("PERBANDINGAN PENGARUH DATA AUGMENTATION")
print("=" * 55)
print(f"{'Model':<40} {'Test Acc':>10}")
print("-" * 55)
print(f"{'U-Net (tanpa augmentasi)':<40} {test_acc_unet:>10.4f}")
print(f"{'U-Net (dengan augmentasi)':<40} {test_acc_unet_aug:>10.4f}")
print(f"{'U-Net + MobileNetV2 (tanpa aug)':<40} {test_acc_mob:>10.4f}")
print("=" * 55)

<h3><font color="blue">Latihan 3:</font></h3>

<ul>
    <li>Mengapa augmentasi <strong>geometrik</strong> (flip, rotasi) harus diterapkan ke gambar <em>dan</em> mask, tetapi augmentasi <strong>warna</strong> (brightness, contrast) hanya ke gambar? Jelaskan apa yang terjadi jika mask ikut diubah brightness-nya.</li>
    <li>Tambahkan augmentasi <strong>random rotation</strong> ke fungsi <code>augment_segmentation</code>. Gunakan <code>tf.image.rot90</code> dengan jumlah rotasi acak (0, 1, 2, atau 3 kali 90°). Pastikan gambar dan mask dirotasi dengan jumlah yang sama!</li>
    <li>Latih model <strong>U-Net + MobileNetV2</strong> dengan dataset augmentasi (<code>train_dataset_aug</code>). Apakah hasilnya lebih baik dibanding tanpa augmentasi?</li>
</ul>

<p><strong>Hint Random Rotation:</strong></p>

```python
# Rotasi kelipatan 90° (diterapkan ke gambar DAN mask)
k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
image = tf.image.rot90(image, k=k)
mask  = tf.image.rot90(mask, k=k)
```


In [ ]:
# Kerjakan Latihan 3 pada cell berikut ini



---
## 4. Evaluasi Mendalam: IoU, Dice Coefficient & Analisis Per-Kelas

<p>Pada segmentation, <strong>accuracy saja tidak cukup</strong> sebagai metrik evaluasi. Mengapa?</p>

<p>Bayangkan sebuah gambar dimana 90% piksel adalah background dan hanya 10% adalah objek (hewan). Model yang <em>selalu memprediksi background</em> akan mendapatkan accuracy 90% — padahal tidak berguna sama sekali! Ini disebut <strong>class imbalance problem</strong>.</p>

<p><strong>Metrik yang lebih informatif:</strong></p>

<table border="1" cellpadding="8" cellspacing="0">
<tr><th>Metrik</th><th>Formula</th><th>Range</th><th>Kegunaan</th></tr>
<tr><td><strong>IoU (Intersection over Union)</strong></td><td>TP / (TP + FP + FN)</td><td>[0, 1]</td><td>Standar utama evaluasi segmentation</td></tr>
<tr><td><strong>Dice Coefficient</strong></td><td>2·TP / (2·TP + FP + FN)</td><td>[0, 1]</td><td>Populer di medical imaging; lebih toleran ke overlap kecil</td></tr>
<tr><td><strong>Pixel Accuracy</strong></td><td>Piksel benar / Total piksel</td><td>[0, 1]</td><td>Sederhana tapi bias ke kelas mayoritas</td></tr>
</table>

<p><strong>Hubungan IoU dan Dice:</strong></p>
<ul>
    <li>Dice selalu ≥ IoU untuk prediksi yang sama</li>
    <li>Jika IoU = 0.5, maka Dice = 0.667</li>
    <li>Keduanya mengukur overlap, tetapi Dice memberi "bobot lebih" pada area yang benar</li>
</ul>


In [ ]:
# Implementasi metrik evaluasi segmentation

def compute_iou_per_class(y_true, y_pred, num_classes=3):
    """
    Hitung IoU (Intersection over Union) untuk setiap kelas.
    Input: y_true dan y_pred berupa array 2D (H×W) dengan nilai integer [0, num_classes-1]
    Output: list IoU per kelas + mean IoU
    """
    iou_per_class = []
    for cls in range(num_classes):
        true_cls = (y_true == cls)
        pred_cls = (y_pred == cls)

        intersection = np.logical_and(true_cls, pred_cls).sum()
        union = np.logical_or(true_cls, pred_cls).sum()

        iou = intersection / (union + 1e-7)  # epsilon untuk hindari div by zero
        iou_per_class.append(iou)

    return iou_per_class, np.mean(iou_per_class)


def compute_dice_per_class(y_true, y_pred, num_classes=3):
    """
    Hitung Dice Coefficient untuk setiap kelas.
    Dice = 2 * |A ∩ B| / (|A| + |B|)
    """
    dice_per_class = []
    for cls in range(num_classes):
        true_cls = (y_true == cls)
        pred_cls = (y_pred == cls)

        intersection = np.logical_and(true_cls, pred_cls).sum()
        total = true_cls.sum() + pred_cls.sum()

        dice = (2.0 * intersection) / (total + 1e-7)
        dice_per_class.append(dice)

    return dice_per_class, np.mean(dice_per_class)


print("Fungsi compute_iou_per_class dan compute_dice_per_class siap!")
print("\nContoh: jika prediksi sempurna → IoU = 1.0, Dice = 1.0")
print("         jika prediksi nol overlap → IoU = 0.0, Dice = 0.0")

In [ ]:
# Evaluasi semua model segmentation dengan IoU dan Dice

class_labels = ['Hewan', 'Background', 'Border']

def evaluate_model_full(model, dataset, model_name, num_batches=10):
    """
    Evaluasi model pada beberapa batch data test.
    Menghitung: pixel accuracy, IoU per-kelas, Dice per-kelas.
    """
    all_iou = []
    all_dice = []
    total_correct = 0
    total_pixels = 0

    for i, (images, masks) in enumerate(dataset):
        if i >= num_batches:
            break

        preds = model.predict(images, verbose=0)
        pred_masks = np.argmax(preds, axis=-1)  # (B, H, W)
        true_masks = masks[:, :, :, 0].numpy()  # (B, H, W)

        for j in range(pred_masks.shape[0]):
            iou_cls, _ = compute_iou_per_class(true_masks[j], pred_masks[j])
            dice_cls, _ = compute_dice_per_class(true_masks[j], pred_masks[j])
            all_iou.append(iou_cls)
            all_dice.append(dice_cls)

            total_correct += (true_masks[j] == pred_masks[j]).sum()
            total_pixels += true_masks[j].size

    avg_iou  = np.mean(all_iou, axis=0)
    avg_dice = np.mean(all_dice, axis=0)
    pixel_acc = total_correct / total_pixels

    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    print(f"  Pixel Accuracy : {pixel_acc:.4f}")
    print(f"  Mean IoU (mIoU): {np.mean(avg_iou):.4f}")
    print(f"  Mean Dice      : {np.mean(avg_dice):.4f}")
    print(f"\n  {'Kelas':<15} {'IoU':>8} {'Dice':>8}")
    print(f"  {'-'*35}")
    for k, label in enumerate(class_labels):
        print(f"  {label:<15} {avg_iou[k]:>8.4f} {avg_dice[k]:>8.4f}")

    return pixel_acc, avg_iou, avg_dice


# Evaluasi ketiga model
print("Mengevaluasi semua model (mungkin butuh beberapa menit)...")

acc1, iou1, dice1 = evaluate_model_full(model_unet,     test_dataset, 'U-Net (Scratch)')
acc2, iou2, dice2 = evaluate_model_full(model_unet_mob, test_dataset, 'U-Net + MobileNetV2')
acc3, iou3, dice3 = evaluate_model_full(model_unet_aug, test_dataset, 'U-Net + Augmentasi')

In [ ]:
# Visualisasi perbandingan metrik per-kelas (bar chart)
x = np.arange(len(class_labels))
width = 0.25

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# --- IoU per kelas ---
bars1 = ax1.bar(x - width, iou1, width, label='U-Net (Scratch)',     color='#FF6B6B', edgecolor='black')
bars2 = ax1.bar(x,         iou2, width, label='U-Net + MobileNetV2', color='#4ECDC4', edgecolor='black')
bars3 = ax1.bar(x + width, iou3, width, label='U-Net + Augmentasi',  color='#45B7D1', edgecolor='black')

ax1.set_title('IoU per Kelas', fontsize=13, fontweight='bold')
ax1.set_xlabel('Kelas')
ax1.set_ylabel('IoU Score')
ax1.set_xticks(x)
ax1.set_xticklabels(class_labels)
ax1.legend(fontsize=9)
ax1.set_ylim(0, 1.0)
ax1.grid(axis='y', alpha=0.3)

# Tambahkan nilai di atas bar
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{height:.3f}', ha='center', va='bottom', fontsize=8)

# --- Dice per kelas ---
bars4 = ax2.bar(x - width, dice1, width, label='U-Net (Scratch)',     color='#FF6B6B', edgecolor='black')
bars5 = ax2.bar(x,         dice2, width, label='U-Net + MobileNetV2', color='#4ECDC4', edgecolor='black')
bars6 = ax2.bar(x + width, dice3, width, label='U-Net + Augmentasi',  color='#45B7D1', edgecolor='black')

ax2.set_title('Dice Coefficient per Kelas', fontsize=13, fontweight='bold')
ax2.set_xlabel('Kelas')
ax2.set_ylabel('Dice Score')
ax2.set_xticks(x)
ax2.set_xticklabels(class_labels)
ax2.legend(fontsize=9)
ax2.set_ylim(0, 1.0)
ax2.grid(axis='y', alpha=0.3)

for bars in [bars4, bars5, bars6]:
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{height:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Perbandingan Metrik Segmentation: IoU & Dice per Kelas',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Ringkasan final
print("\n" + "=" * 65)
print("RINGKASAN FINAL: MEAN IoU & MEAN DICE")
print("=" * 65)
print(f"{'Model':<35} {'mIoU':>8} {'mDice':>8} {'Pix Acc':>9}")
print("-" * 65)
print(f"{'U-Net (Scratch)':<35} {np.mean(iou1):>8.4f} {np.mean(dice1):>8.4f} {acc1:>9.4f}")
print(f"{'U-Net + MobileNetV2':<35} {np.mean(iou2):>8.4f} {np.mean(dice2):>8.4f} {acc2:>9.4f}")
print(f"{'U-Net + Augmentasi':<35} {np.mean(iou3):>8.4f} {np.mean(dice3):>8.4f} {acc3:>9.4f}")
print("=" * 65)

### Confusion Matrix per-Piksel

<p>Kita juga bisa melihat <strong>confusion matrix</strong> pada level piksel — untuk memahami kesalahan model secara lebih detail: kelas mana yang sering tertukar?</p>


In [ ]:
# Confusion Matrix per-piksel
from sklearn.metrics import confusion_matrix
import itertools

def pixel_confusion_matrix(model, dataset, num_batches=10):
    """Hitung confusion matrix pada level piksel dari beberapa batch."""
    all_true = []
    all_pred = []

    for i, (images, masks) in enumerate(dataset):
        if i >= num_batches:
            break
        preds = model.predict(images, verbose=0)
        pred_masks = np.argmax(preds, axis=-1).flatten()
        true_masks = masks[:, :, :, 0].numpy().flatten()
        all_true.extend(true_masks)
        all_pred.extend(pred_masks)

    return confusion_matrix(all_true, all_pred, labels=[0, 1, 2])


def plot_confusion_matrix(cm, classes, title='Confusion Matrix'):
    """Visualisasi confusion matrix dengan warna dan angka."""
    # Normalisasi ke persentase per baris
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax, label='Persentase (%)')

    ax.set_xticks(range(len(classes)))
    ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, fontsize=11)
    ax.set_yticklabels(classes, fontsize=11)

    # Tulis angka di setiap cell
    for i in range(len(classes)):
        for j in range(len(classes)):
            color = 'white' if cm_norm[i, j] > 50 else 'black'
            ax.text(j, i, f'{cm_norm[i,j]:.1f}%\n({cm[i,j]:,})',
                    ha='center', va='center', color=color, fontsize=10)

    ax.set_xlabel('Prediksi', fontsize=12)
    ax.set_ylabel('Ground Truth', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


# Hitung dan tampilkan confusion matrix untuk model terbaik
print("Menghitung confusion matrix (U-Net + MobileNetV2)...")
cm = pixel_confusion_matrix(model_unet_mob, test_dataset)
plot_confusion_matrix(cm, class_labels, 'Pixel Confusion Matrix — U-Net + MobileNetV2')

print("\nInterpretasi: baris = kelas sebenarnya, kolom = prediksi model.")
print("Diagonal = prediksi benar. Off-diagonal = kesalahan model.")
print("Perhatikan kelas mana yang paling sering tertukar!")

<h3><font color="blue">Latihan 4:</font></h3>

<ul>
    <li>Perhatikan hasil mIoU dan mDice di atas. Kelas mana yang memiliki skor terendah? Mengapa menurut Anda kelas tersebut lebih sulit untuk disegmentasi? <em>(Hint: perhatikan proporsi piksel per kelas dari visualisasi distribusi sebelumnya)</em></li>
    <li>Dari confusion matrix, identifikasi kelas yang paling sering <strong>tertukar</strong>. Apakah hal ini masuk akal secara visual? Jelaskan.</li>
    <li>Implementasikan <strong>Dice Loss</strong> sebagai pengganti <code>sparse_categorical_crossentropy</code> dan latih ulang model. Apakah mIoU meningkat?</li>
    <li><strong>(Tantangan)</strong> Hitung <strong>confusion matrix</strong> untuk model U-Net scratch dan bandingkan dengan U-Net + MobileNetV2. Model mana yang lebih sering salah mengklasifikasikan border sebagai background?</li>
</ul>

<p><strong>Hint Dice Loss:</strong></p>

```python
def dice_loss(y_true, y_pred, num_classes=3):
    """
    Dice Loss = 1 - Dice Coefficient.
    y_true: (B, H, W, 1) integer labels
    y_pred: (B, H, W, num_classes) softmax probabilities
    """
    y_true_onehot = tf.one_hot(tf.squeeze(tf.cast(y_true, tf.int32), axis=-1), num_classes)
    smooth = 1e-6
    intersection = tf.reduce_sum(y_true_onehot * y_pred, axis=[1, 2])  # (B, C)
    union = tf.reduce_sum(y_true_onehot + y_pred, axis=[1, 2])         # (B, C)
    dice = (2.0 * intersection + smooth) / (union + smooth)            # (B, C)
    return 1.0 - tf.reduce_mean(dice)

# Penggunaan:
model.compile(optimizer=..., loss=dice_loss, metrics=['accuracy'])
```


In [ ]:
# Kerjakan Latihan 4 pada cell berikut ini



---
## Rangkuman

<table border="1" cellpadding="8" cellspacing="0">
<tr><th>Topik</th><th>Poin Penting</th></tr>
<tr><td><strong>Semantic Segmentation</strong></td><td>Klasifikasi setiap piksel ke kelas tertentu; output berupa mask sekuran gambar input</td></tr>
<tr><td><strong>U-Net</strong></td><td>Arsitektur encoder-decoder dengan skip connections; encoder mengekstrak fitur, decoder merekonstruksi mask</td></tr>
<tr><td><strong>Skip Connections</strong></td><td>Menghubungkan encoder ke decoder pada resolusi sama; mempertahankan detail spasial yang hilang saat downsampling</td></tr>
<tr><td><strong>Pre-trained Backbone</strong></td><td>Gunakan CNN pre-trained (MobileNetV2, ResNet, dll.) sebagai encoder U-Net untuk Transfer Learning pada segmentation</td></tr>
<tr><td><strong>Data Augmentation</strong></td><td>Augmentasi geometrik harus diterapkan ke gambar DAN mask secara konsisten; augmentasi warna hanya ke gambar</td></tr>
<tr><td><strong>IoU (mIoU)</strong></td><td>Metrik evaluasi utama segmentation: Intersection over Union; lebih informatif dari accuracy untuk kelas tidak seimbang</td></tr>
<tr><td><strong>Dice Coefficient</strong></td><td>Alternatif IoU yang lebih toleran; bisa juga digunakan sebagai loss function (Dice Loss = 1 − Dice)</td></tr>
<tr><td><strong>Confusion Matrix</strong></td><td>Visualisasi kesalahan model pada level piksel; menunjukkan kelas mana yang sering tertukar</td></tr>
</table>

---
<center><strong>Selamat Mengerjakan!</strong></center>
